# Exercise 43

**Source:** [LeetCode 185](https://leetcode.com/problems/department-top-three-salaries/?envType=study-plan-v2&envId=top-sql-50) (Hard).

Table: `Employee`
```
+--------------+---------+
| Column Name  | Type    |
+--------------+---------+
| id           | int     |
| name         | varchar |
| salary       | int     |
| departmentId | int     |
+--------------+---------+
```
- `id` is the primary key.
- `departmentId` is a foreign key referencing `id` from `Department`.
- Every row indicates `id`, `name` and Department of an employee and their annual salary.

Table: `Department`
```
+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| id          | int     |
| name        | varchar |
+-------------+---------+
```
- `id` is the primary key.
- `name` is the name of the department.

A person is called **high earner** if their salary is in the top 3 salaries of the department.

**Task.** Return all high earners from each department in any order.

Connect to the database `postgres` using the username `postgres`.

In [1]:
%load_ext sql
%sql postgresql+psycopg://postgres@localhost:5432/postgres

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

Connecting to 'postgresql+psycopg://postgres@localhost:5432/postgres'

### Create new database for the exercise

Make sure first that a database with the same name is removed in case it already exists.

In [2]:
%sql DROP DATABASE IF EXISTS exercise_43 WITH (FORCE);
%sql CREATE DATABASE exercise_43;

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

++
||
++
++

Connect to the database.

*Caution.* No semicolon at the end otherwise it belongs to the name of the database.

In [3]:
%sql postgresql+psycopg://postgres@localhost:5432/exercise_43 --alias exercise_43

Connecting and switching to connection 'exercise_43'

#### Fill the database with content

In [7]:
%%sql
DROP TABLE IF EXISTS Employee;
DROP TABLE IF EXISTS Department;

CREATE TABLE IF NOT EXISTS Department (
    id INTEGER PRIMARY KEY,
    name VARCHAR NOT NULL
);

INSERT INTO Department(id, name)
VALUES
(1, 'IT'),
(2, 'Sales');

CREATE TABLE IF NOT EXISTS Employee (
    id INTEGER PRIMARY KEY,
    name VARCHAR NOT NULL,
    salary INTEGER NOT NULL,
    departmentId INTEGER NOT NULL,
    FOREIGN KEY (departmentId) REFERENCES Department(id)
);

INSERT INTO Employee (id, name, salary, departmentId)
VALUES
(1, 'Joe', 85000, 1),
(2, 'Henry', 80000, 2),
(3, 'Sam', 60000, 2),
(4, 'Max', 90000, 1),
(5, 'Janet', 69000, 1),
(6, 'Randy', 85000, 1),
(7, 'Will', 70000, 1);

Running query in 'exercise_43'

2 rows affected.

7 rows affected.

++
||
++
++

#### Solve the exercise

A person is called **high earner** if their salary is in the top 3 salaries of the department.

**Task.** Return all high earners from each department in any order.

In [8]:
%%sql
WITH SalaryComp AS (
    SELECT
        *, LAG(salary, 1) OVER (Partition BY departmentId ORDER BY salary DESC) prev_salary
    FROM
        Employee
),
CountSalaries AS (
    SELECT
        *, COUNT(CASE WHEN prev_salary IS NOT NULL AND prev_salary > salary THEN 1 END) OVER (PARTITION BY departmentId ORDER BY salary DESC)
    FROM
        SalaryComp
)
SELECT
    D.name AS Department, CS.name AS Employee, CS.Salary
FROM
    Department D LEFT JOIN CountSalaries CS
ON
    D.id = CS.departmentId
WHERE CS.count < 3

Running query in 'exercise_43'

6 rows affected.

department,employee,salary
IT,Max,90000
IT,Joe,85000
IT,Randy,85000
IT,Will,70000
Sales,Henry,80000
Sales,Sam,60000


#### Explanation

Use `WITH ... AS ()` to work with two auxiliary tables.

1) The table `SalaryComp` uses the window function `LAG()` to show the employee's salary and the next highest salary (or null if is the highest salary).
In order to do this for every department separately use `PARTITION BY departmentId`.
The `ORDER BY salary DESC` makes sure that the ordering is w.r.t. descending salary.

In [11]:
%%sql
SELECT
    *, LAG(salary, 1) OVER (Partition BY departmentId ORDER BY salary DESC) prev_salary
FROM
    Employee

Running query in 'exercise_43'

7 rows affected.

id,name,salary,departmentid,prev_salary
4,Max,90000,1,None
1,Joe,85000,1,90000
6,Randy,85000,1,85000
7,Will,70000,1,85000
5,Janet,69000,1,70000
2,Henry,80000,2,None
3,Sam,60000,2,80000


2) The second table is based counts for each department the number of salaries and assigns a number to each salary, where 0 is the highest salary, 1 the second highest salary and so on.
This is done again by a window function using `COUNT(CASE WHEN ... THEN ... END)` and then `OVER (PARTITION BY ...)` in order to do the count for every `departmentId` separately.

In [12]:
%%sql
WITH SalaryComp AS (
    SELECT
        *, LAG(salary, 1) OVER (Partition BY departmentId ORDER BY salary DESC) prev_salary
    FROM
        Employee
)
SELECT
    *, COUNT(CASE WHEN prev_salary IS NOT NULL AND prev_salary > salary THEN 1 END) OVER (PARTITION BY departmentId ORDER BY salary DESC)
FROM
    SalaryComp

Running query in 'exercise_43'

7 rows affected.

id,name,salary,departmentid,prev_salary,count
4,Max,90000,1,None,0
1,Joe,85000,1,90000,1
6,Randy,85000,1,85000,1
7,Will,70000,1,85000,2
5,Janet,69000,1,70000,3
2,Henry,80000,2,None,0
3,Sam,60000,2,80000,1


Afterwards select only the parts of the table where `count < 3` and join with the `Department` table in on `departmentId` in order  to obtain the department names.

#### Close connection

In [13]:
%sql --close exercise_43
%sql postgresql+psycopg://postgres@localhost:5432/postgres
%sql DROP DATABASE IF EXISTS exercise_43;

Switching to connection 'postgresql+psycopg://postgres@localhost:5432/postgres'

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

++
||
++
++

**Well done!** (approved by Leet)